# 11_评估与错误分析

**学习目标：** 计算accuracy和固定40类Macro F1，读混淆矩阵并找到三组真实混淆动作。

适合：读过CS50P但基础不熟、刚开始深度学习。按顺序学习；每课先看讲解、跑示例、做练习，再查答案。


## 本课要点

Accuracy按全部样本计正确比例；Macro F1先对每类算F1再等权平均。混淆矩阵行是真实类、列是预测类。类别不均衡时二者可能相差很大。本课重算已有初版完整验证CSV，结果属于该旧运行。


In [ ]:
from pathlib import Path
import sys
sys.dont_write_bytecode = True
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'tools/learning.py').is_file() and (p / 'config.json').is_file())
sys.path.insert(0, str(ROOT / 'tools'))
import learning as L
OUT = ROOT / 'outputs'
OUT.mkdir(exist_ok=True)


## 用已有完整CSV重算，而不是照抄数字


In [ ]:
get_ipython().run_line_magic('matplotlib', 'inline')
import numpy as np
from analyze_predictions import analyze
run = L.PROJECT / L.CONFIG['reference_run']
report = analyze(run / 'validation.csv', L.PROJECT / 'data/raw/validation/manifest.json')
cm = np.asarray(report['confusion_matrix'])
print('旧运行', run.name, '样本数', report['samples'])
print(report['metrics'])
print('矩阵形状', cm.shape)


## 画混淆矩阵，按数量找错分方向


In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 6))
image = ax.imshow(cm, cmap='Blues')
ax.set_xlabel('predicted class'); ax.set_ylabel('true class')
fig.colorbar(image, ax=ax); plt.show()
wrong = cm.copy()
np.fill_diagonal(wrong, 0)
# 合并两个方向，选三组不同动作对；同一对不会重复列出。
pairs = [(i, j, int(wrong[i,j] + wrong[j,i])) for i in range(40) for j in range(i+1,40)]
top_pairs = sorted(pairs, key=lambda x: x[2], reverse=True)[:3]
for i, j, count in top_pairs:
    print(report['classes'][i], '↔', report['classes'][j], count)


## 为最常混淆的一对显示真实错例


In [ ]:
from IPython.display import display, Image
i, j, _ = top_pairs[0]
error = next(e for e in report['errors'] if {e['label'], e['predicted_class']} == {i, j})
rows = L.manifest('validation')['samples']
index = next(k for k, r in enumerate(rows) if r.get('sample_id', r['file']+'.npy') == error['sample_id'])
print(error['true_name'], '→', error['predicted_name'])
display(Image(filename=str(L.preview('validation', index))))
print('将另外两组也各找一段，写下观察；画面相似是线索，不是已经证明的错误原因。')


## 动手练习

从cm自行计算百分制accuracy与Macro F1，缺少支持/预测的类别F1按0，始终平均40类。填入包含accuracy和macro_f1的字典。

显示“待完成”表示练习尚未作答；代码示例运行成功不等于练习完成。


In [ ]:
your_answer = None  # TODO：对角线TP、行和、列和
if your_answer is None:
    print('待完成：F1=2TP/(真实数量+预测数量)')
else:
    assert abs(your_answer['accuracy'] - report['metrics']['accuracy']) < 1e-6
    assert abs(your_answer['macro_f1'] - report['metrics']['macro_f1']) < 1e-6
    print('练习通过')


## 常见误区

argmax的轴与混淆矩阵方向不能混淆；反复追逐验证集会引入选模偏差。


## 过关检查与选读

独立计算两个指标；三组错例有真实片段；区分推测原因与已经验证的原因。

选做：改变示例中一个参数，先预测现象再运行，并记下原因。

[对应参考答案](../参考答案/11_评估与错误分析.ipynb) · [官方选读](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html)

完成后在README.md#学习记录写下一句自己的解释。
